# Ch 14. 어텐션 메커니즘 — 해설

> 이 노트북은 다섯 연습문제의 엄밀한 해설과 재현 가능한 검증을 포함합니다.

## 문제 1

**문제**: $\mathrm{Var}(QK^\top_{ij}) = d_k$임을 Q, K의 원소가 iid $\mathcal{N}(0,1)$일 때 유도하라.

### 엄밀한 해설

통제 변수: **key dimension d_k**.  측정 지표: **empirical variance of Q dot K**.  해석과 한계: Each product Q_l K_l has mean zero and variance one; independence makes variances add, giving d_k. Monte Carlo error is reported instead of asserted as an exact sample identity.


In [ ]:
import torch
torch.manual_seed(140)
for d in (8,32,128):
 q=torch.randn(30000,d); k=torch.randn(30000,d); empirical=(q*k).sum(1).var().item(); print({'d_k':d,'empirical_variance':empirical,'relative_error':abs(empirical-d)/d})


## 문제 2

**문제**: 인과적 마스크가 없는 어텐션과 있는 어텐션의 출력을 비교하라.

### 엄밀한 해설

통제 변수: **mask: none versus causal**.  측정 지표: **maximum attention mass assigned to future positions and output difference**.  해석과 한계: The mask sets future logits to -infinity before softmax, so causal future mass is exactly zero. Q, K, and V remain identical.


In [ ]:
import torch
torch.manual_seed(141); q=torch.randn(1,1,5,8); k=torch.randn(1,1,5,8); v=torch.randn(1,1,5,8); score=q@k.transpose(-2,-1)/8**.5; future=torch.triu(torch.ones(5,5,dtype=torch.bool),1); a=score.softmax(-1); ac=score.masked_fill(future,float('-inf')).softmax(-1); plain=a@v; causal=ac@v; print({'plain_future_mass':a.masked_select(future).max().item(),'causal_future_mass':ac.masked_select(future).max().item(),'output_l2_difference':(plain-causal).norm().item()}); assert ac.masked_select(future).max()==0


## 문제 3

**문제**: 어텐션 가중치 행렬을 시각화하고, 대각 성분이 큰 이유를 설명하라.

### 엄밀한 해설

통제 변수: **query/key construction: aligned positional features versus shuffled keys**.  측정 지표: **attention matrix and mean diagonal weight**.  해석과 한계: When each query equals its same-position key, self dot products exceed cross-position dot products. Large diagonals are therefore data/model dependent, not guaranteed by attention itself.


In [ ]:
import torch
q=torch.eye(6)*3; k=q.clone(); scores=q@k.T/6**.5; a=scores.softmax(-1); shuffled=a[:,torch.tensor([1,2,3,4,5,0])]; print({'aligned_attention':a.round(decimals=3).tolist(),'aligned_diagonal_mean':a.diag().mean().item(),'shuffled_diagonal_mean':shuffled.diag().mean().item()}); assert a.diag().mean()>shuffled.diag().mean()


## 문제 4

**문제**: 시퀀스 길이 256, 512, 1024, 2048에서 CPU 시간을 측정하고 $O(n^2)$를 검증하라.

### 엄밀한 해설

통제 변수: **sequence length: 256, 512, 1024, 2048**.  측정 지표: **median CPU time and normalized time/n^2**.  해석과 한계: Single-threaded matrix multiplication is timed after warmup. Runtime includes kernel effects, so n^2 element count is also printed as the exact complexity metric.


In [ ]:
import torch
import time, statistics
torch.set_num_threads(1); torch.manual_seed(142); d=16
for n in (256,512,1024,2048):
 q=torch.randn(n,d); k=torch.randn(n,d); _=q@k.T; times=[]
 for _ in range(3): t=time.perf_counter(); _=q@k.T; times.append(time.perf_counter()-t)
 med=statistics.median(times); print({'n':n,'median_seconds':med,'score_elements':n*n,'seconds_per_n2':med/(n*n)})


## 문제 5

**문제**: PyTorch SDPA의 `is_causal=True` 옵션을 사용해 인과적 어텐션을 구현하라.

### 엄밀한 해설

통제 변수: **implementation: PyTorch SDPA is_causal=True versus explicit causal mask**.  측정 지표: **maximum absolute output error**.  해석과 한계: The explicit reference applies the same scale, triangular mask, softmax, and value product. Agreement verifies the SDPA causal option on CPU.


In [ ]:
import torch
torch.manual_seed(143); q=torch.randn(2,3,5,8); k=torch.randn(2,3,5,8); v=torch.randn(2,3,5,8); got=torch.nn.functional.scaled_dot_product_attention(q,k,v,is_causal=True); mask=torch.triu(torch.ones(5,5,dtype=torch.bool),1); score=(q@k.transpose(-2,-1))/8**.5; ref=score.masked_fill(mask,float('-inf')).softmax(-1)@v; err=(got-ref).abs().max().item(); print({'shape':list(got.shape),'max_reference_error':err}); assert err<1e-5
